# Model F — Doctor Summary Generator

Compiles outputs from Models B, C, and D into a concise clinician-readable brief.
No training data required — uses Gemini with strict factual reporting rules.

Clinician-facing only. Reports what was said and what the system computed. Nothing more.

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
import json
import re
import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Optional

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    generation_config=genai.types.GenerationConfig(temperature=0.1, max_output_tokens=512),
    safety_settings=[
        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_ONLY_HIGH"},
    ],
)

In [ ]:
@dataclass
class SessionInput:
    # From Model B
    chief_complaint:         str            = ""
    duration:                str            = ""
    severity:                str            = ""
    body_part:               str            = ""
    associated_symptoms:     List[str]      = field(default_factory=list)
    red_flags_present:       Optional[bool] = None
    additional_observations: str            = ""
    # From Model C
    questions_asked:         List[str]      = field(default_factory=list)
    patient_answers:         List[str]      = field(default_factory=list)
    # From Model D
    priority:                str            = "MEDIUM"
    suspected_issue:         str            = ""
    risk_factors:            List[str]      = field(default_factory=list)
    score_confidence:        float          = 0.0
    # Session metadata
    session_id:              str            = ""
    session_language:        str            = "english"
    patient_age:             Optional[int]  = None
    raw_transcript:          str            = ""


@dataclass
class DoctorBrief:
    session_id:        str
    timestamp:         str
    priority_flag:     str
    suspected_issue:   str
    score_confidence:  float
    narrative_summary: str
    key_findings:      List[str]
    red_flag_note:     str
    conversation_log:  List[dict]
    raw_transcript:    str

In [ ]:
SYSTEM_PROMPT = """\
You are a clinical documentation assistant. Write a patient brief for a doctor.
Rules:
- Report only what the patient said and what the system recorded.
- No clinical interpretation, diagnosis, or advice.
- Use clinical language. Past tense. Concise — readable in under 30 seconds.
- Output valid JSON only matching the schema. No extra text.
"""


def build_prompt(inp: SessionInput) -> str:
    convo = "\n".join(
        f"  Q: {q}\n  A: {a}"
        for q, a in zip(inp.questions_asked, inp.patient_answers)
    ) or "  (none)"

    assoc = ", ".join(inp.associated_symptoms) if inp.associated_symptoms else "none"
    rf    = "YES" if inp.red_flags_present else "No" if inp.red_flags_present is False else "Unknown"

    schema = {
        "narrative_summary": "<2-4 sentence factual summary of what the patient reported>",
        "key_findings":      ["<key fact>"],
        "red_flag_note":     "<one sentence if red flag, else empty string>",
    }

    return f"""\
Session: {inp.session_id or 'N/A'} | Language: {inp.session_language} | Age: {inp.patient_age or 'unknown'}

Model B — Structured symptoms:
  Chief complaint : {inp.chief_complaint or 'not recorded'}
  Body part       : {inp.body_part or 'not recorded'}
  Duration        : {inp.duration or 'not recorded'}
  Severity        : {inp.severity or 'not recorded'}
  Other symptoms  : {assoc}
  Red flag        : {rf}
  Observations    : {inp.additional_observations or 'none'}

Model C — Conversation:
{convo}

Model D — Risk score:
  Priority        : {inp.priority}
  Suspected issue : {inp.suspected_issue or 'not determined'}
  Risk factors    : {', '.join(inp.risk_factors) or 'none'}
  Confidence      : {inp.score_confidence:.2f}

Schema:
{json.dumps(schema, indent=2)}

Return the populated JSON only."""

In [ ]:
def _parse(raw: str) -> dict:
    cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")
    match   = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        raise ValueError("No JSON in response.")
    return json.loads(match.group(0))


def _assemble(inp: SessionInput, gemini_out: dict) -> DoctorBrief:
    return DoctorBrief(
        session_id        = inp.session_id or "N/A",
        timestamp         = datetime.datetime.now().isoformat(),
        priority_flag     = inp.priority,
        suspected_issue   = inp.suspected_issue or "not determined",
        score_confidence  = inp.score_confidence,
        narrative_summary = gemini_out.get("narrative_summary", ""),
        key_findings      = gemini_out.get("key_findings", []),
        red_flag_note     = gemini_out.get("red_flag_note", ""),
        conversation_log  = [
            {"question": q, "answer": a}
            for q, a in zip(inp.questions_asked, inp.patient_answers)
        ],
        raw_transcript    = inp.raw_transcript,
    )


def generate_summary(session_input: SessionInput) -> dict:
    """
    Generate a structured doctor brief from a completed session.
    Returns dict with 'brief' (DoctorBrief) and 'brief_dict'.
    """
    chat = model.start_chat(history=[
        {"role": "user",  "parts": [SYSTEM_PROMPT]},
        {"role": "model", "parts": ["Understood. JSON brief only, no diagnosis."]},
    ])

    try:
        response   = chat.send_message(build_prompt(session_input))
        gemini_out = _parse(response.text)
        success    = True
        error      = None
    except Exception as e:
        gemini_out = {
            "narrative_summary": f"Patient reported {session_input.chief_complaint or 'unspecified complaint'}. Duration: {session_input.duration or 'unknown'}. Severity: {session_input.severity or 'unknown'}.",
            "key_findings":      session_input.risk_factors or [session_input.chief_complaint],
            "red_flag_note":     "Red flag confirmed by system." if session_input.red_flags_present else "",
        }
        success = False
        error   = str(e)

    brief = _assemble(session_input, gemini_out)
    return {"brief": brief, "brief_dict": asdict(brief), "success": success, "error": error}


def summary_from_pipeline(model_b_output: dict, model_c_context,
                           model_d_output: dict, session_id: str = "",
                           session_language: str = "english",
                           patient_age: Optional[int] = None,
                           raw_transcript: str = "") -> dict:
    """Convenience wrapper that accepts Models B, C, and D outputs directly."""
    ext = model_b_output.get("extraction_dict", {})
    scr = model_d_output.get("score_dict", {})
    return generate_summary(SessionInput(
        chief_complaint         = ext.get("chief_complaint", ""),
        duration                = ext.get("duration", ""),
        severity                = ext.get("severity", ""),
        body_part               = ext.get("body_part", ""),
        associated_symptoms     = ext.get("associated_symptoms", []),
        red_flags_present       = ext.get("red_flags_present"),
        additional_observations = ext.get("additional_observations", ""),
        questions_asked         = getattr(model_c_context, "questions_asked", []),
        patient_answers         = getattr(model_c_context, "patient_answers", []),
        priority                = scr.get("priority", "MEDIUM"),
        suspected_issue         = scr.get("suspected_issue", ""),
        risk_factors            = scr.get("risk_factors", []),
        score_confidence        = scr.get("confidence", 0.0),
        session_id              = session_id,
        session_language        = session_language,
        patient_age             = patient_age,
        raw_transcript          = raw_transcript,
    ))

---
## Usage

### Option A — Direct input

In [ ]:
result = generate_summary(SessionInput(
    session_id          = "SESSION-001",
    session_language    = "english",
    patient_age         = 54,
    chief_complaint     = "chest pain",
    body_part           = "chest",
    duration            = "2 hours",
    severity            = "severe",
    associated_symptoms = ["shortness of breath"],
    red_flags_present   = True,
    questions_asked     = ["How severe is the pain?", "Is it spreading anywhere?"],
    patient_answers     = ["About eight out of ten.", "Yes, to my left arm."],
    priority            = "HIGH",
    suspected_issue     = "cardiorespiratory-related complaint",
    risk_factors        = ["severe chest pain", "shortness of breath", "pain radiation to left arm"],
    score_confidence    = 0.95,
    raw_transcript      = "I have had chest pain for two hours and cannot breathe well.",
))
print(json.dumps(result["brief_dict"], indent=2))

### Option B — From pipeline

In [ ]:
# result = summary_from_pipeline(
#     model_b_output   = model_b_result,
#     model_c_context  = context,
#     model_d_output   = model_d_result,
#     session_id       = "SESSION-001",
#     session_language = "english",
#     patient_age      = 54,
#     raw_transcript   = model_a_result["full_text"],
# )
# print(json.dumps(result["brief_dict"], indent=2))